In [1]:
import pandas as pd
import numpy as np
pd.options.display.max_rows = 999

In [2]:
class BuySell:

    def __init__(self, capital):
        self.initial_capital = capital
        self.current_capital = capital
        self.share_amount = 0


    def buyandhold(self, dataframe):
        """

        Args:
            dataframe: (pd.DataFrame) should contains price and directive, date columns

        Returns: (float) profit

        """
        first_price = dataframe.iloc[0].loc['price']
        last_price  = dataframe.iloc[-1].loc['price']

        self.current_capital, self.share_amount = BuySell._buy(self.initial_capital, first_price)
        money, self.share_amount = BuySell._sell(self.share_amount, last_price)
        self.current_capital += money

        profit = self.current_capital - self.initial_capital

        return profit

    @staticmethod
    def _buy(money, price):
        """buys if appliable
        Returns: (float, int) current_money, share_amount"""

        remaining_money = money
        share_amount = 0

        if money > price:
            share_amount = int(money / price)
            remaining_money = money - share_amount * price

        return remaining_money, share_amount

    @staticmethod
    def _sell(share_amount, price):
        """sells all shares :)
        Returns: (float, int) current_money, share_amount"""
        money = share_amount * price
        return money, 0



    def process(self, dataframe):
        """

        Args:
            dataframe: (pd.DataFrame) should contains price and directive, date columns

        Returns: (float) profit

        """
        capital_list = list()
        share_amount_list = list()

        for idx,row in dataframe.iterrows():
            if row['directive'] == 'buy':
                remaining_money, share_amount = BuySell._buy(self.current_capital, row['price'])
                self.current_capital = remaining_money
                self.share_amount += share_amount

            if row['directive'] == 'sell':
                money, share_amount = BuySell._sell(self.share_amount, row['price'])
                self.current_capital += money
                self.share_amount = share_amount

            capital_list.append(self.current_capital)
            share_amount_list.append(self.share_amount)

        dataframe.loc[:,'current_capital'] = capital_list
        dataframe.loc[:,'share_amount'] = share_amount_list
        dataframe.loc[:,'total_capital'] = dataframe['current_capital'] + dataframe['share_amount']*dataframe['price']

        dataframe.loc[:,'profit'] = dataframe['total_capital'] - self.initial_capital


        return dataframe
    
def label_to_directives(row):
    row = row[['pbuy','psell','phold']]
    argmax_idx = np.argmax(row.values)

    if argmax_idx == 0:
        return 'buy'
    if argmax_idx == 1:
        return 'sell'
    return 'hold'

In [3]:
# np.random.seed(42)
# size = 100
# fake_dataframe = pd.DataFrame({'price':np.random.randint(100,150, size=size),
#                                'directive':np.random.choice(['buy','hold','sell'], size=size, replace=True),
#      'name':np.random.choice(['xlp','xlu','xlv','xly','dia','ewa','ewc','ewg','ewh','ewj','eww','spy','xlb','xle','xlf','xli','xlk'], size=size, replace=True)})


In [8]:
initial_capital = 100000
stock_names = ['dia', 'ewa', 'ewc', 'ewg', 'ewh', 'ewj', 'eww', 'spy', 'xlb', 'xle', 'xlf', 'xli', 'xlk', 'xlp', 'xlu', 'xlv', 'xly']
exp_name = 'stock_exp'

final_result_df = None
for stock_name in stock_names:
    path = '../experiment/finance_cnn/{}/{}/prediction_results.csv'.format(exp_name,stock_name)
    dataframe = pd.read_csv(path)
    dataframe['directive'] = dataframe.apply(label_to_directives, axis=1)
    dataframe['price'] = dataframe['raw_adjusted_close'].values
    
    # Buy and Hold strategy
    buyandhold_profit = BuySell(capital=initial_capital).buyandhold(dataframe)
    
    # Our simple strategy
    buysell = BuySell(capital=initial_capital)
    buysell_result_df = buysell.process(dataframe)
    subset = ['name','date','price','directive','current_capital','share_amount','total_capital', 'profit']
#     subset = buysell_result_df.columns
    if final_result_df is None:
        final_result_df = pd.DataFrame(columns=subset)
    
    last_row = buysell_result_df[subset].iloc[-1]
    last_row['bah_profit'] = buyandhold_profit
    
    final_result_df = final_result_df.append(last_row)
    
final_result_df = final_result_df.reset_index(drop=True)
final_result_df

,name,date,price,directive,current_capital,share_amount,total_capital,profit,bah_profit
0,dia,2016-11-17,189.30,hold,150.92,582,110323.52,10323.52,8635.11
1,ewa,2016-11-17,20.20,buy,8.44,5435,109795.44,9795.44,-345.31
2,ewc,2016-11-17,25.42,hold,10.00,4040,102706.80,2706.80,2706.80
3,ewg,2016-11-17,25.09,hold,110703.79,0,110703.79,10703.79,-10167.20
4,ewh,2016-11-17,20.71,hold,20.03,5321,110217.94,10217.94,-4606.00
5,ewj,2016-11-17,50.14,hold,88720.48,0,88720.48,-11279.52,-1298.88
6,eww,2016-11-17,44.21,buy,13.84,1812,80122.36,-19877.64,-18602.40
7,spy,2016-11-17,218.99,hold,11.64,485,106221.79,6221.79,5892.60
8,xlb,2016-11-17,48.39,hold,140163.91,0,140163.91,40163.91,8545.83
9,xle,2016-11-17,70.84,hold,3.16,1529,108317.52,8317.52,3399.47


In [9]:
final_result_df['profit'].mean(),final_result_df['bah_profit'].mean()

(6315.8470588235305, 1239.3835294117691)

In [10]:
initial_capital = 100000
exp_name = 'stress_exp'
final_result_df = None
for i in range(28):
    path = '../experiment/finance_cnn/{}/{}/prediction_results.csv'.format(exp_name,i)
    dataframe = pd.read_csv(path)
    dataframe['directive'] = dataframe.apply(label_to_directives, axis=1)
    dataframe['price'] = dataframe['raw_adjusted_close'].values
    
    # Buy and Hold strategy
    buyandhold_profit = BuySell(capital=initial_capital).buyandhold(dataframe)
    
    # Our simple strategy
    buysell = BuySell(capital=initial_capital)
    buysell_result_df = buysell.process(dataframe)
    subset = ['name','date','price','directive','current_capital','share_amount','total_capital', 'profit']
#     subset = buysell_result_df.columns
    if final_result_df is None:
        final_result_df = pd.DataFrame(columns=subset)
    
    last_row = buysell_result_df[subset].iloc[-1]
    last_row['bah_profit'] = buyandhold_profit
    last_row['deleted_colnum'] = i
    
    final_result_df = final_result_df.append(last_row)
    
final_result_df = final_result_df.reset_index(drop=True)

final_result_df

,name,date,price,directive,current_capital,share_amount,total_capital,profit,bah_profit,deleted_colnum
0,dia,2016-11-17,189.3,sell,104985.91,0,104985.91,4985.91,8635.11,0.0
1,dia,2016-11-17,189.3,hold,78.60,590,111765.60,11765.60,8635.11,1.0
2,dia,2016-11-17,189.3,sell,106037.22,0,106037.22,6037.22,8635.11,2.0
3,dia,2016-11-17,189.3,hold,169.05,528,100119.45,119.45,8635.11,3.0
4,dia,2016-11-17,189.3,sell,97216.56,0,97216.56,-2783.44,8635.11,4.0
5,dia,2016-11-17,189.3,sell,111765.60,0,111765.60,11765.60,8635.11,5.0
6,dia,2016-11-17,189.3,hold,3.69,574,108661.89,8661.89,8635.11,6.0
7,dia,2016-11-17,189.3,hold,96925.70,0,96925.70,-3074.30,8635.11,7.0
8,dia,2016-11-17,189.3,buy,154.68,548,103891.08,3891.08,8635.11,8.0
9,dia,2016-11-17,189.3,sell,108524.16,0,108524.16,8524.16,8635.11,9.0


In [11]:
final_result_df['profit'].mean()

6305.285000000001